## 0. Cargar y preparar `CARPETA_DATOS.zip`

Esta celda está pensada para Google Colab. Selecciona `CARPETA_DATOS.zip` cuando aparezca la ventana de carga.


In [ ]:
# Importamos la herramienta de Colab para subir archivos.
from google.colab import files

# Importamos las bibliotecas que utilizaremos.
from pathlib import Path
import zipfile
import os
import sqlite3
import pandas as pd
import numpy as np

# Subimos el ZIP desde la computadora.
archivos = files.upload()
nombre_zip = next(iter(archivos))

# Extraemos el ZIP sin modificar su contenido.
with zipfile.ZipFile(nombre_zip, "r") as archivo_zip:
    archivo_zip.extractall("/content")

# Definimos la carpeta principal.
carpeta_datos = Path("/content/CARPETA_DATOS")

print("Carpeta disponible:", carpeta_datos.exists())
print("Ruta:", carpeta_datos)
print("Subcarpetas:", sorted(p.name for p in carpeta_datos.iterdir() if p.is_dir()))


Saving CARPETA_DATOS.zip to CARPETA_DATOS.zip
Carpeta disponible: True
Ruta: /content/CARPETA_DATOS
Subcarpetas: ['C5', 'INEGI', 'Kaggle', 'SSC']


# 3.6 Inventario y primera lectura de los datos crudos

Proyecto: **Análisis de hechos de tránsito en la Ciudad de México**

Este notebook sigue la lógica: **leer → inventariar → perfilar → detectar señales → documentar dudas**.

> En esta etapa **no limpiamos ni corregimos** los datos. Conservamos los archivos tal como fueron descargados y generamos evidencia reproducible para el Avance01.

Pregunta de negocio:

**¿En qué alcaldías y horarios de la CDMX se concentra la mayor frecuencia de hechos de tránsito y en cuáles se observa una mayor severidad registrada, considerando las personas lesionadas y fallecidas?**


## 1. Inventario general de archivos

Primero observamos los datos que pudimos obtener. Esto ayuda a identificar formatos, fuentes y archivos disponibles sin alterar los datos.


## 2. Funciones de lectura e inventario

Como existen muchos CSV, definimos funciones que permiten obtener dimensiones y columnas sin modificar los archivos originales.


In [ ]:
def leer_csv_robusto(ruta, nrows=None):
    """
    Intenta leer un CSV conservando la información tal como está.
    No realiza limpieza ni homologación.
    """
    intentos = [
        {"encoding": "utf-8", "sep": ","},
        {"encoding": "utf-8-sig", "sep": ","},
        {"encoding": "latin-1", "sep": ","},
        {"encoding": "utf-8", "sep": ";"},
        {"encoding": "latin-1", "sep": ";"},
    ]

    ultimo_error = None
    for parametros in intentos:
        try:
            return pd.read_csv(ruta, nrows=nrows, low_memory=False, **parametros)
        except Exception as e:
            ultimo_error = e

    raise ultimo_error


def contar_filas_csv(ruta):
    """
    Cuenta líneas del archivo sin cargarlo completo.
    Se resta una línea correspondiente al encabezado.
    """
    with open(ruta, "rb") as f:
        return max(sum(1 for _ in f) - 1, 0)


def ficha_csv(ruta):
    """
    Obtiene filas, columnas y nombres de columnas de un CSV.
    """
    muestra = leer_csv_robusto(ruta, nrows=5)
    return {
        "archivo": ruta.name,
        "filas": contar_filas_csv(ruta),
        "columnas": len(muestra.columns),
        "formato": "CSV",
        "nombres_columnas": muestra.columns.tolist()
    }


# 3. Fuente principal: SSC

Para la pregunta de negocio, la tabla principal es `HechosTransito_SSC.csv`, ya que contiene información del hecho, alcaldía, fecha/hora y variables de lesionados y fallecidos.


In [ ]:
carpeta_ssc = carpeta_datos / "SSC"
archivos_ssc = sorted(carpeta_ssc.glob("*.csv"))

inventario_ssc = []

for ruta in archivos_ssc:
    ficha = ficha_csv(ruta)
    inventario_ssc.append({
        "Archivo o tabla": ficha["archivo"],
        "Filas": ficha["filas"],
        "Columnas": ficha["columnas"],
        "Formato": ficha["formato"]
    })

inventario_ssc = pd.DataFrame(inventario_ssc)
display(inventario_ssc)


,Archivo o tabla,Filas,Columnas,Formato
0,HechosTransito_SSC.csv,134088,26,CSV
1,PersonasLyFporEdad_SSC.csv,158077,23,CSV
2,PersonasLyFporSexo_SSC.csv,144044,24,CSV
3,PersonasLyFporTipoPersona_SSC.csv,107795,24,CSV
4,VehiculosInvolucrados_SSC.csv,215095,26,CSV


### 3.1 Lectura de `HechosTransito_SSC.csv`

La lectura se realiza sin cambiar nombres, categorías, faltantes ni tipos.


In [ ]:
ruta_ssc = carpeta_ssc / "HechosTransito_SSC.csv"
hechos = leer_csv_robusto(ruta_ssc)

print(type(hechos))
print("Forma:", hechos.shape)
print("Columnas:")
print(hechos.columns.tolist())

# Muestra legible para evidencia.
display(hechos.head(5))


<class 'pandas.core.frame.DataFrame'>
Forma: (134079, 26)
Columnas:
['fecha_evento', 'hora_evento', 'tipo_evento', 'fecha_captura', 'folio', 'latitud', 'longitud', 'punto_1', 'punto_2', 'colonia', 'alcaldia', 'zona_vial', 'sector', 'unidad_a_cargo', 'tipo_de_interseccion', 'interseccion_semaforizada', 'clasificacion_de_la_vialidad', 'sentido_de_circulacion', 'dia', 'prioridad', 'origen', 'unidad_medica_de_apoyo', 'matricula_unidad_medica', 'trasladado_lesionados', 'personas_fallecidas', 'personas_lesionadas']


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,personas_fallecidas,personas_lesionadas
0,2020-04-06,12:50:00,CHOQUE,2020-04-17,BJ/200406/03499,19.368116,-99.142903,EJE 7 SUR,ANTILLAS,PORTALES NTE,...,EJE VIAL,P-O,Lunes,BAJA,RADIO,PC,NaN,NO,0,1
1,2020-04-06,18:31:00,CHOQUE,2020-04-17,C5/200406/05748,19.301142,-99.115521,CALZ DEL HUESO,RANCHO COLORADO,COAPA STA CECILIA,...,VIA PRIMARIA,O-P,Lunes,BAJA,911 CDMX,ERUM,NaN,NO,0,1
2,2020-04-06,18:39:00,CHOQUE,2020-04-17,C5/200406/05802,19.476843,-99.092207,EJE 5 NTE,AV GRAN CANAL DEL DESAGUE,JOSE MA MORELOS Y PAVON,...,EJE VIAL,P-O,Lunes,BAJA,911 CDMX,PARTICULAR,NaN,NO,0,1
3,2020-04-06,11:38:00,DERRAPADO,2020-04-17,IZ/200406/03058,19.298474,-98.984670,EJE 10 SUR,AV SAN FCO,SAN FCO TLALTENCO,...,EJE VIAL,NO-SP,Lunes,MEDIA,RADIO,PC,NaN,SI,0,1
4,2020-04-06,13:31:00,DERRAPADO,2020-04-17,C5/200406/03762,19.436170,-99.204754,AV HOMERO,SOFOCLES,LOS MORALES,...,VIA PRIMARIA,P-O,Lunes,BAJA,911 CDMX,SEGURO,NaN,NO,0,1


### 3.2 Tipos y perfil de todas las columnas


In [ ]:
print("Tipos de datos asignados por Pandas:")
display(hechos.dtypes.rename("tipo_aparente").to_frame())

# Construimos el perfil básico, siguiendo la lógica del notebook de ejemplo.
perfil_ssc = pd.DataFrame({
    "tipo_aparente": hechos.dtypes.astype(str),
    "presentes": hechos.notna().sum(),
    "faltantes": hechos.isna().sum(),
    "distintos": hechos.nunique(dropna=True)
})

perfil_ssc["porcentaje_faltante"] = (
    perfil_ssc["faltantes"] / len(hechos) * 100
).round(2)

perfil_ssc = perfil_ssc.sort_values(
    ["porcentaje_faltante", "faltantes"],
    ascending=False
)

display(perfil_ssc)


Tipos de datos asignados por Pandas:


,tipo_aparente
fecha_evento,object
hora_evento,object
tipo_evento,object
fecha_captura,object
folio,object
latitud,float64
longitud,float64
punto_1,object
punto_2,object
colonia,object


,tipo_aparente,presentes,faltantes,distintos,porcentaje_faltante
matricula_unidad_medica,object,51892,82187,4326,61.30
unidad_a_cargo,object,68333,65746,7667,49.04
fecha_captura,object,95879,38200,1608,28.49
origen,object,99036,35043,29,26.14
interseccion_semaforizada,object,99037,35042,3,26.14
clasificacion_de_la_vialidad,object,99037,35042,8,26.14
sentido_de_circulacion,object,99037,35042,17,26.14
hora_evento,object,129070,5009,1444,3.74
latitud,float64,134072,7,79094,0.01
longitud,float64,134075,4,76157,0.00


### 3.3 Valores faltantes

La siguiente salida sirve como evidencia para señalar qué columnas presentan más valores faltantes. No se reemplaza ni elimina ninguno.


In [ ]:
faltantes_ssc = pd.DataFrame({
    "faltantes": hechos.isna().sum(),
    "porcentaje": (hechos.isna().mean() * 100).round(2)
})

faltantes_ssc = faltantes_ssc[
    faltantes_ssc["faltantes"] > 0
].sort_values("faltantes", ascending=False)

display(faltantes_ssc)


,faltantes,porcentaje
matricula_unidad_medica,82187,61.30
unidad_a_cargo,65746,49.04
fecha_captura,38200,28.49
origen,35043,26.14
interseccion_semaforizada,35042,26.14
sentido_de_circulacion,35042,26.14
clasificacion_de_la_vialidad,35042,26.14
hora_evento,5009,3.74
latitud,7,0.01
longitud,4,0.00


### 3.4 Duplicados exactos y folios repetidos

Una repetición de `folio` **no se considera automáticamente un error**. Primero debemos comprobar qué representa cada fila y si un mismo hecho puede aparecer más de una vez.


In [ ]:
duplicados_exactos = int(hechos.duplicated().sum())

print("Filas completamente idénticas:", duplicados_exactos)

if "folio" in hechos.columns:
    folios_repetidos = int(hechos.duplicated(subset=["folio"]).sum())
    folios_unicos = int(hechos["folio"].nunique(dropna=True))

    print("Folios distintos:", folios_unicos)
    print("Repeticiones adicionales de folio:", folios_repetidos)

    # Mostramos algunos folios que aparecen más de una vez.
    ejemplo_folios = hechos.loc[
        hechos.duplicated(subset=["folio"], keep=False)
    ].sort_values("folio")

    display(ejemplo_folios.head(20))


Filas completamente idénticas: 0
Folios distintos: 126472
Repeticiones adicionales de folio: 7606


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,personas_fallecidas,personas_lesionadas
17300,2019-03-15,08:00:00,CHOQUE,2019-03-18,1097536,19.442119,-99.151057,VIOLETA,JUAN ALDAMA,BUENAVISTA,...,NaN,NaN,Viernes,BAJA,NaN,ERUM,NaN,NO,0,1
37755,2019-03-15,08:29:00,CHOQUE,2019-03-18,1097536,19.458862,-99.163293,CIRCUITO BICENTENARIO,JAZMIN,TLATILCO,...,NaN,NaN,Viernes,MEDIA,NaN,PC,NaN,SI,0,1
17412,2019-03-21,08:50:00,ATROPELLADO,2019-03-22,1136192,19.383058,-99.158867,SAN BORJA,ANAXAGORAS,NARVARTE PTE,...,NaN,NaN,Jueves,MEDIA,NaN,PARAMEDIC,NaN,SI,0,1
17411,2019-03-21,08:40:00,ATROPELLADO,2019-03-22,1136192,19.435464,-99.185574,AV HOMERO,PETRARCA,POLANCO 5A SECC,...,NaN,NaN,Jueves,BAJA,NaN,CRUZ ROJA,NaN,NO,0,1
5355,2020-01-28,14:38:00,ATROPELLADO,2020-01-29,2404440,19.306032,-99.218507,LINEA 1,LINEA 4,EMILIO PORTES GIL,...,VIA SECUNDARIA,O-P,Martes,BAJA,PM,ERUM,NaN,NO,0,1
32917,2020-01-28,14:46:00,DERRAPADO,2020-01-29,2404440,19.425184,-99.201425,ANILLO PERIFERICO,PEDREGAL,BOSQUE DE CHAPULTEPEC 2A SECC,...,VAC ANULAR,N-S,Martes,BAJA,PM,CRUZ ROJA,NaN,NO,0,1
5414,2020-01-31,00:20:00,DERRAPADO,2020-02-02,2411167,19.323210,-98.958188,AUT FED MEXICO PUEBLA,AV AGRICULTURA,STA CATARINA,...,ACCESO CARRETERO,P-O,Viernes,BAJA,PM,PC,NaN,NO,0,1
32943,2020-01-31,01:39:00,CHOQUE,2020-02-02,2411167,19.278543,-99.167336,AV INSURGENTES,CALZ DE TLALPAN,LA JOYA,...,VIA PRIMARIA,N-S,Viernes,BAJA,PM,URGENCIAS BASICAS,NaN,NO,0,1
23715,2018-01-01,17:37:00,ATROPELLADO,2018-03-01,314955,19.454556,-99.129457,ALUMINIO,CALZ DE GUADALUPE,MAZA,...,NaN,NaN,Lunes,BAJA,NaN,CRUZ ROJA,NaN,NO,0,1
13270,2018-01-03,18:50:00,ATROPELLADO,2018-04-01,314955,19.319486,-99.245254,AV SAN BERNABE,CRUZ VERDE,BARROS SIERRA,...,NaN,NaN,Miércoles,ALTA,NaN,CCO,NaN,NO,1,0


### 3.5 Revisión de la variable `alcaldia`

La pregunta de negocio utiliza las alcaldías como unidad territorial. Por ello es importante observar las categorías exactamente como aparecen en el archivo crudo.


In [ ]:
if "alcaldia" in hechos.columns:
    print("Cantidad de valores distintos:", hechos["alcaldia"].nunique(dropna=True))

    conteo_alcaldias = hechos["alcaldia"].value_counts(dropna=False).to_frame("registros")
    display(conteo_alcaldias)


Cantidad de valores distintos: 18


,registros
alcaldia,
CUAUHTEMOC,19669
IZTAPALAPA,19262
GUSTAVO A MADERO,14762
BENITO JUAREZ,10753
VENUSTIANO CARRANZA,10055
MIGUEL HIDALGO,9825
COYOACAN,9339
ALVARO OBREGON,7752
TLALPAN,7562


### 3.6 Fechas y horas: revisión sin sobrescribir el origen

Creamos series temporales únicamente para comprobar qué valores pueden interpretarse como fechas u horas. La tabla `hechos` permanece sin cambios.


In [ ]:
if "fecha_evento" in hechos.columns:
    fecha_temporal = pd.to_datetime(hechos["fecha_evento"], errors="coerce", dayfirst=True)

    print("Valores originales no faltantes:", hechos["fecha_evento"].notna().sum())
    print("Fechas no convertibles:", (
        hechos["fecha_evento"].notna() & fecha_temporal.isna()
    ).sum())
    print("Fecha mínima interpretable:", fecha_temporal.min())
    print("Fecha máxima interpretable:", fecha_temporal.max())

if "hora_evento" in hechos.columns:
    hora_temporal = pd.to_datetime(
        hechos["hora_evento"].astype("string"),
        errors="coerce"
    )

    print("\nHoras no convertibles:", (
        hechos["hora_evento"].notna() & hora_temporal.isna()
    ).sum())

    print("\nEjemplos de hora:")
    display(hechos[["hora_evento"]].head(10))


Valores originales no faltantes: 134079
Fechas no convertibles: 81256
Fecha mínima interpretable: 2018-01-01 00:00:00
Fecha máxima interpretable: 2023-12-12 00:00:00


/tmp/ipykernel_961/2497952325.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  hora_temporal = pd.to_datetime(



Horas no convertibles: 0

Ejemplos de hora:


,hora_evento
0,12:50:00
1,18:31:00
2,18:39:00
3,11:38:00
4,13:31:00
5,15:31:00
6,10:19:00
7,19:05:00
8,08:58:00
9,13:52:00


### 3.7 Variables de severidad: lesionados y fallecidos

Solo describimos lo que contiene el archivo. No calculamos todavía un indicador compuesto de severidad ni corregimos valores.


In [ ]:
columnas_severidad = [
    c for c in ["personas_lesionadas", "personas_fallecidas"]
    if c in hechos.columns
]

for columna in columnas_severidad:
    temporal = pd.to_numeric(hechos[columna], errors="coerce")

    print("\n---", columna, "---")
    print("Tipo original:", hechos[columna].dtype)
    print("Valores no convertibles:", (
        hechos[columna].notna() & temporal.isna()
    ).sum())
    print(temporal.describe())

    # Mostramos los valores mayores únicamente como señal para revisión.
    muestra_extremos = (
        hechos.assign(valor_numerico=temporal)
        .nlargest(10, "valor_numerico")
        [["folio", columna, "valor_numerico"]]
        if "folio" in hechos.columns else
        hechos.assign(valor_numerico=temporal).nlargest(10, "valor_numerico")[[columna, "valor_numerico"]]
    )
    display(muestra_extremos)



--- personas_lesionadas ---
Tipo original: int64
Valores no convertibles: 0
count    134079.000000
mean          1.163411
std           0.625307
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          25.000000
Name: personas_lesionadas, dtype: float64


,folio,personas_lesionadas,valor_numerico
81174,C5/20220328/02894,25,25
6967,1145720,22,22
124708,C5/20230908/02856,21,21
20788,713915,20,20
122045,C5/20230807/00886,20,20
12847,976410,18,18
72620,C5/20211211/02636,17,17
83985,C5/20220430/02002,17,17
91260,C5/20220723/00523,17,17
129627,C5/20231108/01275,17,17



--- personas_fallecidas ---
Tipo original: int64
Valores no convertibles: 0
count    134079.000000
mean          0.019451
std           0.149814
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          10.000000
Name: personas_fallecidas, dtype: float64


,folio,personas_fallecidas,valor_numerico
11215,805319,10,10
13890,386285,5,5
24280,377643,5,5
48756,2245234,5,5
11294,811523,4,4
38937,1586199,4,4
107972,C5/20230207/04216,4,4
112,C5/200415/00272,3,3
18432,1577723,3,3
19621,1857505,3,3


## 4. Granularidad de las tablas de SSC

Las tablas auxiliares no deben unirse directamente sin revisar sus claves. Comparamos filas, folios y repeticiones para obtener evidencia de su posible granularidad.


In [ ]:
resumen_granularidad_ssc = []

for ruta in archivos_ssc:
    df = leer_csv_robusto(ruta)

    fila = {
        "archivo": ruta.name,
        "filas": len(df),
        "columnas": df.shape[1],
        "duplicados_exactos": int(df.duplicated().sum())
    }

    if "folio" in df.columns:
        fila["folios_distintos"] = int(df["folio"].nunique(dropna=True))
        fila["repeticiones_adicionales_folio"] = int(
            df.duplicated(subset=["folio"]).sum()
        )
    else:
        fila["folios_distintos"] = np.nan
        fila["repeticiones_adicionales_folio"] = np.nan

    resumen_granularidad_ssc.append(fila)

    # Liberamos memoria antes de leer el siguiente archivo.
    del df

resumen_granularidad_ssc = pd.DataFrame(resumen_granularidad_ssc)
display(resumen_granularidad_ssc)


,archivo,filas,columnas,duplicados_exactos,folios_distintos,repeticiones_adicionales_folio
0,HechosTransito_SSC.csv,134079,26,0,126472,7606
1,PersonasLyFporEdad_SSC.csv,158077,23,3498,126253,31824
2,PersonasLyFporSexo_SSC.csv,144044,24,0,126472,17571
3,PersonasLyFporTipoPersona_SSC.csv,107795,24,0,99000,8795
4,VehiculosInvolucrados_SSC.csv,215079,26,0,126472,88606


# 5. C5 – Incidentes viales

Los archivos están separados por periodos. Primero inventariamos su estructura y después revisamos una muestra de sus columnas.


In [ ]:
carpeta_c5 = carpeta_datos / "C5"
archivos_c5 = sorted(carpeta_c5.glob("*.csv"))

inventario_c5 = []

for ruta in archivos_c5:
    ficha = ficha_csv(ruta)
    inventario_c5.append({
        "Archivo o tabla": ficha["archivo"],
        "Filas": ficha["filas"],
        "Columnas": ficha["columnas"],
        "Formato": ficha["formato"]
    })

inventario_c5 = pd.DataFrame(inventario_c5)
display(inventario_c5)


,Archivo o tabla,Filas,Columnas,Formato
0,inViales_2014_2015.csv,356072,17,CSV
1,inViales_2016_2018.csv,664111,17,CSV
2,inViales_2019_2021.csv,590636,17,CSV
3,inViales_2022_2024.csv,504261,17,CSV


In [ ]:
# Leemos solo una pequeña muestra de cada archivo para comparar sus columnas.
estructuras_c5 = {}

for ruta in archivos_c5:
    muestra = leer_csv_robusto(ruta, nrows=5)
    estructuras_c5[ruta.name] = muestra.columns.tolist()

for archivo, columnas in estructuras_c5.items():
    print("\n", archivo)
    print(columnas)



 inViales_2014_2015.csv
['folio', 'fecha_creacion', 'hora_creacion', 'dia_semana', 'fecha_cierre', 'hora_cierre', 'tipo_incidente_c4', 'incidente_c4', 'alcaldia_inicio', 'codigo_cierre', 'clas_con_f_alarma', 'tipo_entrada', 'alcaldia_cierre', 'alcaldia_catalogo', 'colonia_catalogo', 'longitud', 'latitud']

 inViales_2016_2018.csv
['folio', 'fecha_creacion', 'hora_creacion', 'dia_semana', 'fecha_cierre', 'hora_cierre', 'tipo_incidente_c4', 'incidente_c4', 'alcaldia_inicio', 'codigo_cierre', 'clas_con_f_alarma', 'tipo_entrada', 'alcaldia_cierre', 'alcaldia_catalogo', 'colonia_catalogo', 'longitud', 'latitud']

 inViales_2019_2021.csv
['folio', 'fecha_creacion', 'hora_creacion', 'dia_semana', 'fecha_cierre', 'hora_cierre', 'tipo_incidente_c4', 'incidente_c4', 'alcaldia_inicio', 'codigo_cierre', 'clas_con_f_alarma', 'tipo_entrada', 'alcaldia_cierre', 'alcaldia_catalogo', 'colonia_catalogo', 'longitud', 'latitud']

 inViales_2022_2024.csv
['folio', 'fecha_creacion', 'hora_creacion', 'dia_s

### 5.1 Perfil de cada archivo C5

Esta parte puede tardar un poco porque lee los cuatro archivos completos. Se utiliza para obtener faltantes, duplicados y cobertura temporal como evidencia.


In [ ]:
resumen_c5 = []

for ruta in archivos_c5:
    print("Revisando:", ruta.name)
    df = leer_csv_robusto(ruta)

    fila = {
        "archivo": ruta.name,
        "filas": len(df),
        "columnas": df.shape[1],
        "duplicados_exactos": int(df.duplicated().sum())
    }

    if "folio" in df.columns:
        fila["folios_distintos"] = int(df["folio"].nunique(dropna=True))
        fila["repeticiones_adicionales_folio"] = int(
            df.duplicated(subset=["folio"]).sum()
        )

    if "fecha_creacion" in df.columns:
        fecha = pd.to_datetime(df["fecha_creacion"], errors="coerce", dayfirst=True)
        fila["fecha_minima"] = fecha.min()
        fila["fecha_maxima"] = fecha.max()
        fila["fechas_no_convertibles"] = int(
            (df["fecha_creacion"].notna() & fecha.isna()).sum()
        )

    resumen_c5.append(fila)
    del df

resumen_c5 = pd.DataFrame(resumen_c5)
display(resumen_c5)


Revisando: inViales_2014_2015.csv
Revisando: inViales_2016_2018.csv
Revisando: inViales_2019_2021.csv
Revisando: inViales_2022_2024.csv


/tmp/ipykernel_961/1929574754.py:21: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  fecha = pd.to_datetime(df["fecha_creacion"], errors="coerce", dayfirst=True)


,archivo,filas,columnas,duplicados_exactos,folios_distintos,repeticiones_adicionales_folio,fecha_minima,fecha_maxima,fechas_no_convertibles
0,inViales_2014_2015.csv,356072,17,0,356072,0,2014-01-01,2015-12-12,217300
1,inViales_2016_2018.csv,664111,17,0,664108,3,2016-01-01,2018-12-12,403716
2,inViales_2019_2021.csv,590636,17,0,590627,9,2019-01-01,2021-12-12,359883
3,inViales_2022_2024.csv,504261,17,0,504261,0,2021-12-29,2024-02-29,0


In [ ]:
# Revisamos faltantes del archivo C5 más reciente como muestra reproducible.
ruta_c5_reciente = carpeta_c5 / "inViales_2022_2024.csv"
c5_reciente = leer_csv_robusto(ruta_c5_reciente)

perfil_c5_reciente = pd.DataFrame({
    "tipo_aparente": c5_reciente.dtypes.astype(str),
    "presentes": c5_reciente.notna().sum(),
    "faltantes": c5_reciente.isna().sum(),
    "distintos": c5_reciente.nunique(dropna=True)
})

perfil_c5_reciente["porcentaje_faltante"] = (
    perfil_c5_reciente["faltantes"] / len(c5_reciente) * 100
).round(2)

display(perfil_c5_reciente.sort_values("faltantes", ascending=False))

# Campos de alcaldía presentes.
columnas_alcaldia = [c for c in c5_reciente.columns if "alcaldia" in c.lower()]
print("Columnas relacionadas con alcaldía:", columnas_alcaldia)


,tipo_aparente,presentes,faltantes,distintos,porcentaje_faltante
colonia_catalogo,object,493085,11176,1404,2.22
alcaldia_catalogo,object,503707,554,16,0.11
alcaldia_inicio,object,504226,35,16,0.01
alcaldia_cierre,object,504226,35,16,0.01
tipo_entrada,object,504256,5,9,0.00
folio,object,504261,0,504261,0.00
fecha_creacion,object,504261,0,792,0.00
dia_semana,object,504261,0,7,0.00
hora_creacion,object,504261,0,83841,0.00
incidente_c4,object,504261,0,16,0.00


Columnas relacionadas con alcaldía: ['alcaldia_inicio', 'alcaldia_cierre', 'alcaldia_catalogo']


# 6. INEGI – ATUS

INEGI contiene archivos anuales, catálogos, diccionario de datos y metadatos. Para el inventario contamos los archivos anuales sin unirlos ni modificarlos.


In [ ]:
carpeta_inegi = (
    carpeta_datos / "INEGI" /
    "conjunto_de_datos_atus_anual_csv" /
    "conjunto_de_datos"
)

archivos_inegi = sorted(carpeta_inegi.glob("atus_anual_*.csv"))

print("Archivos anuales encontrados:", len(archivos_inegi))
print("Primero:", archivos_inegi[0].name)
print("Último:", archivos_inegi[-1].name)

inventario_inegi = []

for ruta in archivos_inegi:
    muestra = leer_csv_robusto(ruta, nrows=5)
    inventario_inegi.append({
        "archivo": ruta.name,
        "filas": contar_filas_csv(ruta),
        "columnas": len(muestra.columns),
        "formato": "CSV"
    })

inventario_inegi = pd.DataFrame(inventario_inegi)

display(inventario_inegi)
print("Total de registros anuales:", inventario_inegi["filas"].sum())


Archivos anuales encontrados: 29
Primero: atus_anual_1997.csv
Último: atus_anual_2025.csv


,archivo,filas,columnas,formato
0,atus_anual_1997.csv,248114,46,CSV
1,atus_anual_1998.csv,262687,46,CSV
2,atus_anual_1999.csv,285494,46,CSV
3,atus_anual_2000.csv,311938,46,CSV
4,atus_anual_2001.csv,364869,46,CSV
5,atus_anual_2002.csv,399002,46,CSV
6,atus_anual_2003.csv,424490,46,CSV
7,atus_anual_2004.csv,443607,46,CSV
8,atus_anual_2005.csv,452233,46,CSV
9,atus_anual_2006.csv,471272,46,CSV


Total de registros anuales: 10999723


### 6.1 Comparación de estructura entre años

No asumimos que todos los años tengan exactamente las mismas columnas.


In [ ]:
columnas_por_anio = {}

for ruta in archivos_inegi:
    muestra = leer_csv_robusto(ruta, nrows=5)
    columnas_por_anio[ruta.name] = set(muestra.columns)

base = columnas_por_anio[archivos_inegi[0].name]

comparacion_estructura = []

for archivo, columnas in columnas_por_anio.items():
    comparacion_estructura.append({
        "archivo": archivo,
        "numero_columnas": len(columnas),
        "faltan_respecto_1997": sorted(base - columnas),
        "adicionales_respecto_1997": sorted(columnas - base)
    })

comparacion_estructura = pd.DataFrame(comparacion_estructura)
display(comparacion_estructura)


,archivo,numero_columnas,faltan_respecto_1997,adicionales_respecto_1997
0,atus_anual_1997.csv,46,[],[]
1,atus_anual_1998.csv,46,[],[]
2,atus_anual_1999.csv,46,[],[]
3,atus_anual_2000.csv,46,[],[]
4,atus_anual_2001.csv,46,[],[]
5,atus_anual_2002.csv,46,[],[]
6,atus_anual_2003.csv,46,[],[]
7,atus_anual_2004.csv,46,[],[]
8,atus_anual_2005.csv,46,[],[]
9,atus_anual_2006.csv,46,[],[]


### 6.2 Diccionario y catálogos de INEGI

Mostramos estos recursos porque algunas variables están codificadas y no deben interpretarse sin consultar su documentación.


In [ ]:
base_inegi = carpeta_datos / "INEGI" / "conjunto_de_datos_atus_anual_csv"

catalogos_inegi = sorted((base_inegi / "catalogos").glob("*.csv"))
diccionarios_inegi = sorted((base_inegi / "diccionario_de_datos").glob("*.csv"))
metadatos_inegi = sorted((base_inegi / "metadatos").glob("*"))

print("Catálogos:")
for r in catalogos_inegi:
    print("-", r.name)

print("\nDiccionario:")
for r in diccionarios_inegi:
    print("-", r.name)

print("\nMetadatos:")
for r in metadatos_inegi:
    print("-", r.name)

if diccionarios_inegi:
    diccionario_inegi = leer_csv_robusto(diccionarios_inegi[0])
    display(diccionario_inegi.head(20))


Catálogos:
- tc_dia.csv
- tc_edad.csv
- tc_entidad.csv
- tc_hora.csv
- tc_minuto.csv
- tc_municipio.csv
- tc_periodo_mes.csv

Diccionario:
- diccionario_de_datos_atus_anual_1997_2025.csv

Metadatos:
- metadatos_atus_anual_1997_2025.txt


,COLUMNA,DESCRIPCION,TIPO_DATO,LONGITUD,CODIGO_VALIDO
0,COBERTURA,Área geográfica a la que están referidos los i...,varchar,200.0,NaN
1,CVEGEO,Clave geoestadística,char,2.0,01001 a 32058
2,ID_ENTIDAD,Clave de la entidad federativa según el Catálo...,char,2.0,01-32
3,ID_MUNICIPIO,Clave del municipio según el Catálogo de Entid...,char,3.0,001-999
4,ANIO,Los cuatro dígitos correspondientes al año en ...,int,NaN,1997-2025
5,MES,Correspondiente al mes de referencia en que oc...,varchar,2.0,01-12
6,ID_HORA,La hora (sin los minutos) en que ocurrió el ac...,int,NaN,0-23
7,ID_MINUTO,"Los minutos en que ocurrió el accidente, con r...",int,NaN,0-59
8,ID_DIA,Número correspondiente al día del mes en que o...,varchar,2.0,0-31
9,DIASEMANA,El día de la semana en que ocurrió el accidente,varchar,20.0,"Lunes-Domingo, Certificado cero, No especificado"


# 7. Kaggle – Accidentes viales en México

La carpeta incluye archivos anuales y una base SQLite `accidentes.db`. La revisamos como una fuente separada y no asumimos que pueda sumarse directamente con INEGI.


In [ ]:
carpeta_kaggle = carpeta_datos / "Kaggle" / "archive"
ruta_db = carpeta_kaggle / "accidentes.db"

print("Base SQLite disponible:", ruta_db.exists())

if ruta_db.exists():
    conexion = sqlite3.connect(ruta_db)

    tablas = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
        conexion
    )
    display(tablas)

    resumen_tablas_sqlite = []

    for tabla in tablas["name"]:
        filas = pd.read_sql_query(
            f'SELECT COUNT(*) AS n FROM "{tabla}"',
            conexion
        )["n"].iloc[0]

        columnas = pd.read_sql_query(
            f'PRAGMA table_info("{tabla}")',
            conexion
        )

        resumen_tablas_sqlite.append({
            "tabla": tabla,
            "filas": int(filas),
            "columnas": len(columnas)
        })

    resumen_tablas_sqlite = pd.DataFrame(resumen_tablas_sqlite)
    display(resumen_tablas_sqlite)

    conexion.close()


Base SQLite disponible: True


,name
0,accidentes
1,entidades
2,municipios


,tabla,filas,columnas
0,accidentes,9943594,36
1,entidades,32,2
2,municipios,2502,2


In [ ]:
# Localizamos los CSV anuales de Kaggle.
archivos_kaggle = sorted(
    carpeta_kaggle.rglob("atus_anual_*.csv")
)

# Evitamos incluir accidentalmente archivos que no sean los anuales del conjunto.
print("Archivos anuales encontrados:", len(archivos_kaggle))

inventario_kaggle = []

for ruta in archivos_kaggle:
    muestra = leer_csv_robusto(ruta, nrows=5)
    inventario_kaggle.append({
        "archivo": ruta.name,
        "filas": contar_filas_csv(ruta),
        "columnas": len(muestra.columns),
        "formato": "CSV"
    })

inventario_kaggle = pd.DataFrame(inventario_kaggle)
display(inventario_kaggle)

if not inventario_kaggle.empty:
    print("Total de registros:", inventario_kaggle["filas"].sum())


Archivos anuales encontrados: 26


,archivo,filas,columnas,formato
0,atus_anual_1997.csv,248114,45,CSV
1,atus_anual_1998.csv,262687,45,CSV
2,atus_anual_1999.csv,285494,45,CSV
3,atus_anual_2000.csv,311938,45,CSV
4,atus_anual_2001.csv,364869,45,CSV
5,atus_anual_2002.csv,399002,45,CSV
6,atus_anual_2003.csv,424490,45,CSV
7,atus_anual_2004.csv,443607,45,CSV
8,atus_anual_2005.csv,452233,45,CSV
9,atus_anual_2006.csv,471272,45,CSV


Total de registros: 9943594


# 8. Tabla de evidencia para el documento

Esta tabla reúne algunos hallazgos que pueden utilizarse en la sección 3.6. Son **señales de revisión**, no reglas de limpieza.


In [ ]:
evidencias = []

# SSC
evidencias.append({
    "Fuente": "SSC",
    "Archivo": "HechosTransito_SSC.csv",
    "Señal": "Filas completamente idénticas",
    "Resultado": int(hechos.duplicated().sum()),
    "Interpretación inicial": "Revisar; no eliminar automáticamente."
})

if "folio" in hechos.columns:
    evidencias.append({
        "Fuente": "SSC",
        "Archivo": "HechosTransito_SSC.csv",
        "Señal": "Repeticiones adicionales de folio",
        "Resultado": int(hechos.duplicated(subset=["folio"]).sum()),
        "Interpretación inicial": "Un folio repetido no implica por sí solo un duplicado."
    })

if "alcaldia" in hechos.columns:
    evidencias.append({
        "Fuente": "SSC",
        "Archivo": "HechosTransito_SSC.csv",
        "Señal": "Valores distintos en alcaldia",
        "Resultado": int(hechos["alcaldia"].nunique(dropna=True)),
        "Interpretación inicial": "Revisar las categorías exactamente como fueron registradas."
    })

# C5
if "resumen_c5" in globals():
    for _, r in resumen_c5.iterrows():
        evidencias.append({
            "Fuente": "C5",
            "Archivo": r["archivo"],
            "Señal": "Cobertura observada en fecha_creacion",
            "Resultado": f'{r.get("fecha_minima", "")} a {r.get("fecha_maxima", "")}',
            "Interpretación inicial": "Comparar con el periodo indicado en el nombre del archivo."
        })

evidencias = pd.DataFrame(evidencias)
display(evidencias)


,Fuente,Archivo,Señal,Resultado,Interpretación inicial
0,SSC,HechosTransito_SSC.csv,Filas completamente idénticas,0,Revisar; no eliminar automáticamente.
1,SSC,HechosTransito_SSC.csv,Repeticiones adicionales de folio,7606,Un folio repetido no implica por sí solo un du...
2,SSC,HechosTransito_SSC.csv,Valores distintos en alcaldia,18,Revisar las categorías exactamente como fueron...
3,C5,inViales_2014_2015.csv,Cobertura observada en fecha_creacion,2014-01-01 00:00:00 a 2015-12-12 00:00:00,Comparar con el periodo indicado en el nombre ...
4,C5,inViales_2016_2018.csv,Cobertura observada en fecha_creacion,2016-01-01 00:00:00 a 2018-12-12 00:00:00,Comparar con el periodo indicado en el nombre ...
5,C5,inViales_2019_2021.csv,Cobertura observada en fecha_creacion,2019-01-01 00:00:00 a 2021-12-12 00:00:00,Comparar con el periodo indicado en el nombre ...
6,C5,inViales_2022_2024.csv,Cobertura observada en fecha_creacion,2021-12-29 00:00:00 a 2024-02-29 00:00:00,Comparar con el periodo indicado en el nombre ...


# 9. Inventario de columnas para anexar

La guía solicita cubrir **todas las columnas**. Esta celda genera un inventario técnico de la fuente principal con tipo aparente, faltantes, valores distintos y ejemplos observados.

El significado de cada variable debe completarse con el diccionario/metadatos oficiales cuando exista; no se inventa a partir del nombre.


In [ ]:
inventario_columnas_ssc = []

for columna in hechos.columns:
    valores = hechos[columna].dropna()
    ejemplos = valores.astype(str).drop_duplicates().head(3).tolist()

    inventario_columnas_ssc.append({
        "Columna": columna,
        "Tipo aparente": str(hechos[columna].dtype),
        "Presentes": int(hechos[columna].notna().sum()),
        "Faltantes": int(hechos[columna].isna().sum()),
        "Valores distintos": int(hechos[columna].nunique(dropna=True)),
        "Ejemplos observados": " | ".join(ejemplos)
    })

inventario_columnas_ssc = pd.DataFrame(inventario_columnas_ssc)
display(inventario_columnas_ssc)


,Columna,Tipo aparente,Presentes,Faltantes,Valores distintos,Ejemplos observados
0,fecha_evento,object,134079,0,2191,2020-04-06 | 2020-04-07 | 2020-04-08
1,hora_evento,object,129070,5009,1444,12:50:00 | 18:31:00 | 18:39:00
2,tipo_evento,object,134079,0,6,CHOQUE | DERRAPADO | ATROPELLADO
3,fecha_captura,object,95879,38200,1608,2020-04-17 | 2020-04-18 | 2020-04-20
4,folio,object,134078,1,126472,BJ/200406/03499 | C5/200406/05748 | C5/200406/...
5,latitud,float64,134072,7,79094,19.368116 | 19.301142 | 19.476843
6,longitud,float64,134075,4,76157,-99.142903 | -99.115521 | -99.092207
7,punto_1,object,134079,0,9250,EJE 7 SUR | CALZ DEL HUESO | EJE 5 NTE
8,punto_2,object,134075,4,16349,ANTILLAS | RANCHO COLORADO | AV GRAN CANAL DEL...
9,colonia,object,134079,0,4250,PORTALES NTE | COAPA STA CECILIA | JOSE MA MOR...


# 10. Exportar las evidencias

Guardamos las tablas de perfilamiento como CSV. Esto no modifica los datos crudos; únicamente crea productos derivados para documentar el análisis.


In [ ]:
carpeta_salida = Path("/content/evidencias_avance1")
carpeta_salida.mkdir(exist_ok=True)

inventario_archivos.to_csv(
    carpeta_salida / "inventario_general_archivos.csv",
    index=False,
    encoding="utf-8-sig"
)

inventario_ssc.to_csv(
    carpeta_salida / "inventario_ssc.csv",
    index=False,
    encoding="utf-8-sig"
)

perfil_ssc.to_csv(
    carpeta_salida / "perfil_columnas_hechos_ssc.csv",
    encoding="utf-8-sig"
)

inventario_columnas_ssc.to_csv(
    carpeta_salida / "inventario_columnas_hechos_ssc.csv",
    index=False,
    encoding="utf-8-sig"
)

inventario_c5.to_csv(
    carpeta_salida / "inventario_c5.csv",
    index=False,
    encoding="utf-8-sig"
)

inventario_inegi.to_csv(
    carpeta_salida / "inventario_inegi.csv",
    index=False,
    encoding="utf-8-sig"
)

inventario_kaggle.to_csv(
    carpeta_salida / "inventario_kaggle.csv",
    index=False,
    encoding="utf-8-sig"
)

evidencias.to_csv(
    carpeta_salida / "evidencias_primera_lectura.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos generados en:", carpeta_salida)
print("\nProductos:")
for ruta in sorted(carpeta_salida.iterdir()):
    print("-", ruta.name)


Archivos generados en: /content/evidencias_avance1

Productos:
- evidencias_primera_lectura.csv
- inventario_c5.csv
- inventario_columnas_hechos_ssc.csv
- inventario_general_archivos.csv
- inventario_inegi.csv
- inventario_kaggle.csv
- inventario_ssc.csv
- perfil_columnas_hechos_ssc.csv


## Cierre

Con este perfilamiento podemos documentar:

- qué archivos recolectamos como equipo
- qué representa cada tabla de manera preliminar
- sus dimensiones y estructura
- los tipos aparentes
- valores faltantes
- duplicados exactos y claves repetidas
- categorías que requieren revisión
- diferencias de cobertura temporal
- diferencias de granularidad
- campos que podrían relacionar tablas
- y dudas que deberán resolverse antes de la preparación de datos

**En esta etapa no se eliminan registros, no se rellenan faltantes, no se homologan categorías y no se sobrescriben los datos crudos.**

Las decisiones detalladas de limpieza y transformación se documentarán en la etapa correspondiente del proyecto.
